# EISForge → AutoREC in-memory integration

This small workflow generates one synthetic EIS sample with EISForge and passes it directly to AutoREC without writing intermediate CSV files.

In [ ]:
# Configure native scientific libraries before importing AutoEIS-dependent modules.
from autorec.runtime import configure_autorec_runtime

configure_autorec_runtime(thread_count=1, warmup_autoeis=True, suppress_tf_logs=True)

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from eisforge import DataGen
from autorec.data_preparation import EISDataPrep

In [ ]:
# EISForge requires an output directory, but both exports are disabled.
# The temporary directory is removed automatically when this block exits.
with TemporaryDirectory(prefix="eisforge-autorec-") as temp_dir:
    generator = DataGen(
        random_ecm_circuit="R1-[P2,R3]",
        output_dir=Path(temp_dir),
        n_random_candidates=20,
        max_selected_curves=3,
        fim_fit_ecm=False,
        drop_pp_series=False,
        drop_invalid_ecms=False,
        excluded_simplified_ecms=(),
        verbose=False,
    )
    eisforge_rows, batch_details = generator.generate_data(
        target_num=1,
        max_batches=6,
        seed_start=7,
        export_summary=False,
        export_samples=False,
        live_plot=False,
    )
    frequency = generator.random_ecm_freq.copy()

    # Guard the example's no-intermediate-CSV requirement.
    assert not list(Path(temp_dir).rglob("*.csv"))

assert len(eisforge_rows) == 1
eisforge_rows[["relabel_ecm", "final_Z"]]

In [ ]:
# AutoREC calculates its normal thresholds and flattened EIS features from
# EISForge's final circuit and impedance arrays entirely in memory.
data_prep = EISDataPrep.from_eisforge(
    eisforge_rows,
    frequency,
    evaluation=True,
)
autorec_dataset = data_prep.dataset

assert len(autorec_dataset) == len(eisforge_rows)
assert set(EISDataPrep.EVAL_REQUIRED_COLUMNS).issubset(autorec_dataset.columns)
autorec_dataset[["sub_id", "true_circuit", "chi_thresh", "r2_thresh"]]